### Structured Output

In [1]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

True

In [2]:
model = init_chat_model(model="groq:llama-3.1-8b-instant")

### Pydantic

- Pydantic provides rich feature set with field validation, description and nested structures.

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str = Field(description="The title of the movie")
    year:int = Field(description="This year the movie was released")
    director:str = Field(description="The director of the movie")
    rating:float = Field(description="The movie's rating out of 10")

In [4]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002A90E27AF90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002A90E27BCB0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'pro

In [6]:
# Model without any schema attached
model.invoke("Provide me details about movie Inception")

AIMessage(content='**Inception (2010)**\n\nDirected by Christopher Nolan and starring Leonardo DiCaprio, Joseph Gordon-Levitt, Ellen Page, Tom Hardy, Ken Watanabe, Dileep Rao, Cillian Murphy, Tom Berenger, and Marion Cotillard, **Inception** is a thought-provoking science fiction action film that delves into the concept of shared dreaming.\n\n**Plot**\n\nThe movie revolves around Cobb (Leonardo DiCaprio), a skilled thief who specializes in entering people\'s dreams and stealing their secrets. Cobb is offered a chance to redeem himself by performing a task known as "inception" - planting an idea in someone\'s mind instead of stealing one. Cobb\'s employer, Saito (Ken Watanabe), wants him to convince Robert Fischer (Cillian Murphy), the son of a dying business magnate, to dissolve his father\'s company.\n\nCobb assembles a team of experts: Arthur (Joseph Gordon-Levitt), a point man; Ariadne (Ellen Page), an architect who designs the dreamscapes; Eames (Tom Hardy), a forger who can impers

In [5]:
# Model is attached with a specific schema, so the output will be following that schema
model_with_structure.invoke("Provide me details about movie Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.5)

### Message output alongside parsed structure

In [7]:

from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title:str = Field(..., description="The title of the movie")  # ... is basically telling model to provide this title strictly
    year:int = Field(..., description="This year the movie was released")
    director:str = Field(..., description="The director of the movie")
    rating:float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)
response = model_with_structure.invoke("Provide me details about movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'xqdx0xcb3', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.5,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 290, 'total_tokens': 323, 'completion_time': 0.043481007, 'completion_tokens_details': None, 'prompt_time': 0.025534692, 'prompt_tokens_details': None, 'queue_time': 0.050299987, 'total_time': 0.069015699}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f32fb-d0b0-70a1-ab16-9aa24e698b29-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Christopher Nolan', 'rating': 8.5, 'title': 'Inception', 'year': 2010}, 'id': 'xqdx0xcb3', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 290, 'output_tokens': 33, 't

### Nested Structure in Pydantic

In [12]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str = Field(..., description="Actor's Name")
    role:str = Field(..., description="Actor's role in the movie")

class MovieDetails(BaseModel):
    title:str = Field(..., description="The title of the movie")
    year:int = Field(..., description="This year the movie was released")
    director:str = Field(..., description="The director of the movie")
    rating:float = Field(..., description="The movie's rating out of 10")
    cast:list[Actor] = Field(..., description="Actors name in the movie")
    genre:list[str] = Field(..., description="movies genre")
    budget:float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails, include_raw=True)
response = model_with_structure.invoke("Provide me details about movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '8rbgytc7t', 'function': {'arguments': '{"budget":160,"cast":[{"name":"Leonardo DiCaprio","role":"Cobb"},{"name":"Joseph Gordon-Levitt","role":"Arthur"},{"name":"Ellen Page","role":"Ariadne"},{"name":"Tom Hardy","role":"Eames"},{"name":"Ken Watanabe","role":"Saito"},{"name":"Dileep Rao","role":"Yusuf"},{"name":"Cillian Murphy","role":"Robert Fischer"},{"name":"Tom Berenger","role":"Proctor"},{"name":"Pete Postlethwaite","role":"Miles"},{"name":"Marion Cotillard","role":"Mal"},{"name":"Michael Caine","role":"Eames"}],"director":"Christopher Nolan","genre":["Action","Sci-Fi","Thriller"],"rating":8.8,"title":"Inception","year":2010}', 'name': 'MovieDetails'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 233, 'prompt_tokens': 387, 'total_tokens': 620, 'completion_time': 0.307099325, 'completion_tokens_details': None, 'prompt_time': 0.024811717, 'prompt_tokens_details': None, 'queue_time

### TypedDict

- TypedDict provides a simple alternative using python built-in typing, ideal when you don't need runtime validation

In [17]:
from typing_extensions import TypedDict, Annotated

class MovieAnnotated(TypedDict):
    """A movie with details."""
    title:  Annotated[str, ..., "The title of the movie"] 
    year:  Annotated[int, ..., "This year the movie was released"]
    director:  Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_with_typedict = model.with_structured_output(MovieAnnotated, include_raw=True)
response = model_with_typedict.invoke("Provide me details about movie Avengers")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'd06b6pb6p', 'function': {'arguments': '{"director":"Joss Whedon","rating":8,"title":"Avengers","year":2012}', 'name': 'MovieAnnotated'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 297, 'total_tokens': 331, 'completion_time': 0.043843279, 'completion_tokens_details': None, 'prompt_time': 0.019905046, 'prompt_tokens_details': None, 'queue_time': 0.159214054, 'total_time': 0.063748325}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f333b-03cd-7df0-8a80-41772eea8232-0', tool_calls=[{'name': 'MovieAnnotated', 'args': {'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}, 'id': 'd06b6pb6p', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 297, 'output_tokens': 34, 't

In [21]:
class Actor(TypedDict):
    name: Annotated[str, ..., "The Name of the Actor"] 
    role: Annotated[str, ..., "The Role of the Actor"] 

class MovieTypedDict(TypedDict):
    title: Annotated[str, ..., "The title of the movie"] 
    year: Annotated[int, ..., "This year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]
    cast: Annotated[list[Actor], ..., "Actors name in the movie"]
    genre: Annotated[list[str], ..., "movies genre"] 
    budget: Annotated[float, ..., "Budget in millions USD"] 

model_with_typedict = model.with_structured_output(MovieTypedDict, include_raw=True)
response = model_with_typedict.invoke("Provide me details about movie Avengers")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'y1k174c62', 'function': {'arguments': '{"budget":220,"cast":[{"name":"Robert Downey Jr.","role":"Tony Stark/Iron Man"},{"name":"Chris Evans","role":"Steve Rogers/Captain America"},{"name":"Mark Ruffalo","role":"Bruce Banner/Hulk"},{"name":"Chris Hemsworth","role":"Thor"},{"name":"Scarlett Johansson","role":"Natasha Romanoff/Black Widow"},{"name":"Jeremy Renner","role":"Clint Barton/Hawkeye"},{"name":"Tom Hiddleston","role":"Loki"}],"director":"Joss Whedon","genre":["Action","Adventure","Sci-Fi"],"rating":8,"title":"Avengers","year":2012}', 'name': 'MovieTypedDict'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 140, 'prompt_tokens': 709, 'total_tokens': 849, 'completion_time': 0.186994543, 'completion_tokens_details': None, 'prompt_time': 0.061699054, 'prompt_tokens_details': None, 'queue_time': 0.050807934, 'total_time': 0.248693597}, 'model_name': 'llama-3.1-8b-instant', 'system_f

In [ ]:
# Important function to know what's models profile 
model.profile

{'name': 'Llama 3.1 8B Instant',
 'release_date': '2024-07-23',
 'last_updated': '2024-07-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 131072,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

### Dataclass

In [1]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

c:\Users\viren\generative-ai-implementations\langchain\agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')